# SECTION 0: AAPL DWA Exposure-Wagebill Bubble Chart

Visualize Apple's DWAs by AI exposure and wagebill, showing employee counts as bubble sizes.

In [105]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from pathlib import Path

# === Define base directory ===
try:
    BASE = Path(__file__).resolve().parent
except NameError:
    BASE = Path.cwd()

# === Load AAPL DWA data ===
aapl_path = BASE / "output/Tables/AAPL_dwa_clustered.parquet"
aapl_dwa = pd.read_parquet(aapl_path)

# Rename columns to match TX convention for easier code reuse
aapl_dwa = aapl_dwa.rename(columns={
    "dwa_id": "DWA_ID",
    "dwa_description": "DWA_TITLE",
    "n_employees": "position_count",
    "total_wagebill": "DWA_WAGEBILL_TOTAL"
})

print(f"Loaded AAPL DWA data: {len(aapl_dwa):,} DWAs")
print(f"Exposure range: {aapl_dwa['exposure'].min():.3f} to {aapl_dwa['exposure'].max():.3f}")
print(f"Total wagebill: ${aapl_dwa['DWA_WAGEBILL_TOTAL'].sum():,.2f}")

# === Filter: exposed DWAs with >= $10M wagebill ===
dwa_exposed = aapl_dwa[
    (aapl_dwa["exposure"] >= 0.4) &
    (aapl_dwa["DWA_WAGEBILL_TOTAL"] >= 1e7)
].copy()

print(f"\nFiltered to {len(dwa_exposed):,} DWAs with exposure >= 0.4 and wagebill >= $10M")

# Convert wagebill to millions
dwa_exposed["wagebill_million"] = dwa_exposed["DWA_WAGEBILL_TOTAL"] / 1e6

# === Interactive Bubble Chart ===
fig = px.scatter(
    dwa_exposed,
    x="exposure",
    y="wagebill_million",
    size="position_count",
    color="wagebill_million",
    hover_data={
        "DWA_ID": True,
        "DWA_TITLE": True,
        "position_count": ":,",
        "wagebill_million": ":.2f",
        "exposure": ":.3f"
    },
    labels={
        "exposure": "AI Augmentation Potential",
        "wagebill_million": "Wagebill (Million $)",
        "position_count": "Employees",
        "DWA_ID": "DWA ID",
        "DWA_TITLE": "DWA Title"
    },
    color_continuous_scale="Blues",
    size_max=30,
    title=(
        "AAPL: DWA Exposure & Wagebill Analysis"
        "<br><sub>Bubble size = Number of employees performing the activity</sub>"
    )
)

# === Layout: FORCE WHITE BACKGROUND ===
fig.update_layout(
    template="plotly_white",      # critical: removes gray panel
    width=1400,
    height=800,
    title_font_size=18,
    title_font_color="gray",
    title_x=0.05,
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="closest",
    font=dict(size=12)
)

# === Axes styling ===
fig.update_xaxes(
    showgrid=False,
    showline=False,
    zeroline=False,
    title_font_size=14
)

fig.update_yaxes(
    showgrid=False,
    showline=False,
    zeroline=False,
    title_font_size=14
)

# === Border color logic (gray → white with wagebill) ===
wagebill_normalized = (
    (dwa_exposed["wagebill_million"] - dwa_exposed["wagebill_million"].min()) /
    (dwa_exposed["wagebill_million"].max() - dwa_exposed["wagebill_million"].min())
)

border_colors = [
    f"rgb({int(128 + 127*v)},{int(128 + 127*v)},{int(128 + 127*v)})"
    for v in wagebill_normalized
]

# === Update traces ===
fig.update_traces(
    hovertemplate=(
        "<b>%{customdata[1]}</b><br>"
        "DWA ID: %{customdata[0]}<br>"
        "Exposure: %{x:.3f}<br>"
        "Wagebill: $%{y:.2f}M<br>"
        "Employees: %{customdata[2]:,}<br>"
        "<extra></extra>"
    ),
    marker=dict(
        line=dict(width=1.2, color=border_colors),
        opacity=0.8
    )
)

# === Summary statistics ===
print(f"\n{'='*80}")
print(f"AAPL DWA EXPOSURE SUMMARY")
print(f"{'='*80}")
print(f"DWAs displayed: {len(dwa_exposed):,}")
print(f"Total employees: {dwa_exposed['position_count'].sum():,}")
print(f"Total wagebill: ${dwa_exposed['DWA_WAGEBILL_TOTAL'].sum():,.2f}")
print(f"Avg exposure: {dwa_exposed['exposure'].mean():.3f}")
print(f"{'='*80}")

Loaded AAPL DWA data: 2,054 DWAs
Exposure range: 0.060 to 0.996
Total wagebill: $27,800,086,970.90

Filtered to 275 DWAs with exposure >= 0.4 and wagebill >= $10M

AAPL DWA EXPOSURE SUMMARY
DWAs displayed: 275
Total employees: 2,842,283
Total wagebill: $15,868,411,264.24
Avg exposure: 0.517


---
# SECTION 1: Automation Scenario Analysis

Perform scenario analysis on AAPL DWAs to estimate potential automation savings at different rates.

In [106]:
import numpy as np

# === Automation Scenario Analysis ===
print("="*80)
print("AUTOMATION SCENARIO ANALYSIS - AAPL")
print("="*80)

# Sort DWAs by exposure score (descending) - only consider those with wagebill >= $10M
df_sorted = aapl_dwa[aapl_dwa['DWA_WAGEBILL_TOTAL'] >= 10e6].sort_values('exposure', ascending=False).copy()

print(f"\nConsidering {len(df_sorted):,} DWAs with wagebill >= $10M")
print(f"Total wagebill of these DWAs: ${df_sorted['DWA_WAGEBILL_TOTAL'].sum():,.2f}")

# Define scenarios
top_n_dwas_list = [5, 10, 15, 20]
automation_rates = [0.05, 0.15, 0.25]  # 5%, 15%, 25%

# Results storage
results = []

for top_n in top_n_dwas_list:
    # Get top N DWAs
    top_dwas = df_sorted.head(top_n)
    total_exposed_wb = top_dwas['DWA_WAGEBILL_TOTAL'].sum()
    avg_exposure = top_dwas['exposure'].mean()
    
    for auto_rate in automation_rates:
        savings = total_exposed_wb * auto_rate
        
        results.append({
            'Top_N_DWAs': top_n,
            'Automation_Rate': f"{int(auto_rate*100)}%",
            'Exposed_Wagebill': total_exposed_wb,
            'Avg_Exposure': avg_exposure,
            'Potential_Savings': savings
        })

# Create results DataFrame
df_results = pd.DataFrame(results)

print("\n" + "="*80)
print("SCENARIO RESULTS")
print("="*80)

for top_n in top_n_dwas_list:
    subset = df_results[df_results['Top_N_DWAs'] == top_n]
    print(f"\n📊 Top {top_n} Most Exposed DWAs (Wagebill >= $10M)")
    print(f"   Total exposed wagebill: ${subset['Exposed_Wagebill'].iloc[0]:,.2f}")
    print(f"   Average exposure score: {subset['Avg_Exposure'].iloc[0]:.3f}")
    print(f"   Savings scenarios:")
    
    for _, row in subset.iterrows():
        print(f"      - {row['Automation_Rate']} automation: ${row['Potential_Savings']:,.2f}")

print("\n" + "="*80)

AUTOMATION SCENARIO ANALYSIS - AAPL

Considering 415 DWAs with wagebill >= $10M
Total wagebill of these DWAs: $25,394,907,970.65

SCENARIO RESULTS

📊 Top 5 Most Exposed DWAs (Wagebill >= $10M)
   Total exposed wagebill: $259,170,069.02
   Average exposure score: 0.772
   Savings scenarios:
      - 5% automation: $12,958,503.45
      - 15% automation: $38,875,510.35
      - 25% automation: $64,792,517.25

📊 Top 10 Most Exposed DWAs (Wagebill >= $10M)
   Total exposed wagebill: $407,942,557.64
   Average exposure score: 0.755
   Savings scenarios:
      - 5% automation: $20,397,127.88
      - 15% automation: $61,191,383.65
      - 25% automation: $101,985,639.41

📊 Top 15 Most Exposed DWAs (Wagebill >= $10M)
   Total exposed wagebill: $529,116,334.45
   Average exposure score: 0.735
   Savings scenarios:
      - 5% automation: $26,455,816.72
      - 15% automation: $79,367,450.17
      - 25% automation: $132,279,083.61

📊 Top 20 Most Exposed DWAs (Wagebill >= $10M)
   Total exposed wageb

In [107]:
# === Create Visualization of Results ===

# Format the savings in millions for better readability
df_results['Savings_Millions'] = df_results['Potential_Savings'] / 1e6
df_results['Exposed_WB_Millions'] = df_results['Exposed_Wagebill'] / 1e6

# Create interactive bar chart
fig = go.Figure()

colors_dict = {'5%': "#7598f7", '15%': "#3e6ef0", '25%': "#1f49ba"}

for auto_rate in ['5%', '15%', '25%']:
    subset = df_results[df_results['Automation_Rate'] == auto_rate]
    
    fig.add_trace(go.Bar(
        x=subset['Top_N_DWAs'],
        y=subset['Savings_Millions'],
        name=f'{auto_rate} Automation',
        marker_color=colors_dict[auto_rate],
        text=[f'${val:.1f}M' for val in subset['Savings_Millions']],
        textposition='auto',
        hovertemplate='<b>Top %{x} DWAs</b><br>' +
                      f'{auto_rate} Automation<br>' +
                      'Potential Savings: $%{y:.2f}M<br>' +
                      '<extra></extra>'
    ))

fig.update_layout(
    title='AAPL Automation Scenario Analysis<br><sub>Potential Wagebill Savings by Top Exposed DWAs</sub>',
    xaxis_title='Number of Top Exposed DWAs (Wagebill >= $10M)',
    yaxis_title='Potential Savings ($ Millions)',
    barmode='group',
    hovermode='closest',
    template='plotly_white',
    height=500,
    xaxis=dict(tickmode='array', tickvals=top_n_dwas_list),
    font=dict(size=12)
)

fig.show()

print("\n✅ Scenario analysis complete!")


✅ Scenario analysis complete!


In [108]:
# === Create 4 Muted Bubble Plots in 2x2 Grid ===
from plotly.subplots import make_subplots

print("\n" + "="*80)
print("GENERATING SCENARIO-SPECIFIC VISUALIZATIONS (2x2 GRID)")
print("="*80)

# Prepare base data - filter for exposure >= 0.6 and wagebill >= $10M
dwa_plot = aapl_dwa[
    (aapl_dwa["exposure"] >= 0.6) &
    (aapl_dwa["DWA_WAGEBILL_TOTAL"] >= 1e7)
].copy()

dwa_plot["wagebill_million"] = dwa_plot["DWA_WAGEBILL_TOTAL"] / 1e6

print(f"\nFiltered to {len(dwa_plot):,} DWAs with exposure >= 0.6 and wagebill >= $10M")

# Get the sorted high-exposure DWAs (wagebill >= $10M)
df_sorted_plot = aapl_dwa[aapl_dwa['DWA_WAGEBILL_TOTAL'] >= 10e6].sort_values('exposure', ascending=False)

# Create subplots: 2 rows x 2 columns
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'Top 5 Most Exposed DWAs',
        f'Top 10 Most Exposed DWAs',
        f'Top 15 Most Exposed DWAs',
        f'Top 20 Most Exposed DWAs'
    ],
    horizontal_spacing=0.12,
    vertical_spacing=0.15
)

# Create 4 plots for different top N scenarios
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
top_n_list = [5, 10, 15, 20]

for idx, (top_n, pos) in enumerate(zip(top_n_list, positions)):
    row, col = pos
    
    # Get top N DWA IDs
    top_n_ids = set(df_sorted_plot.head(top_n)['DWA_ID'].values)
    
    # Split data into highlighted and other
    dwa_highlighted = dwa_plot[dwa_plot['DWA_ID'].isin(top_n_ids)].copy()
    dwa_other = dwa_plot[~dwa_plot['DWA_ID'].isin(top_n_ids)].copy()
    
    # Add gray bubbles first (so they're behind)
    fig.add_trace(
        go.Scatter(
            x=dwa_other['exposure'],
            y=dwa_other['wagebill_million'],
            mode='markers',
            name='Other DWAs',
            marker=dict(
                size=dwa_other['position_count'] / dwa_other['position_count'].max() * 30,
                color='#d1d5db',
                line=dict(width=1, color='white'),
                opacity=0.6
            ),
            customdata=dwa_other[['DWA_ID', 'DWA_TITLE', 'position_count']].values,
            hovertemplate="<b>%{customdata[1]}</b><br>" +
                          "DWA ID: %{customdata[0]}<br>" +
                          "Exposure: %{x:.3f}<br>" +
                          "Wagebill: $%{y:.2f}M<br>" +
                          "Employees: %{customdata[2]:,}<br>" +
                          "<extra></extra>",
            showlegend=(idx == 0),
            legendgroup='other'
        ),
        row=row, col=col
    )
    
    # Add dark blue highlighted bubbles on top
    fig.add_trace(
        go.Scatter(
            x=dwa_highlighted['exposure'],
            y=dwa_highlighted['wagebill_million'],
            mode='markers',
            name=f'Top {top_n} Exposed',
            marker=dict(
                size=dwa_highlighted['position_count'] / dwa_highlighted['position_count'].max() * 30,
                color='#1e3a8a',
                line=dict(width=1.2, color='white'),
                opacity=0.9
            ),
            customdata=dwa_highlighted[['DWA_ID', 'DWA_TITLE', 'position_count']].values,
            hovertemplate="<b>%{customdata[1]}</b><br>" +
                          "DWA ID: %{customdata[0]}<br>" +
                          "Exposure: %{x:.3f}<br>" +
                          "Wagebill: $%{y:.2f}M<br>" +
                          "Employees: %{customdata[2]:,}<br>" +
                          "<extra></extra>",
            showlegend=(idx == 0),
            legendgroup='highlighted'
        ),
        row=row, col=col
    )
    
    # Print summary for this scenario
    top_n_data = df_sorted_plot.head(top_n)
    print(f"✓ Top {top_n} DWAs - Total exposed wagebill: ${top_n_data['DWA_WAGEBILL_TOTAL'].sum():,.2f}")

# Update layout
fig.update_layout(
    title_text='AAPL Automation Scenarios: DWA Exposure Analysis (Exposure >= 0.6)<br><sub>Dark blue = DWAs considered for automation | Gray = Other DWAs</sub>',
    title_font_size=18,
    title_font_color="gray",
    title_x=0.5,
    title_y=0.98,
    title_xanchor='center',
    title_yanchor='top',
    height=1000,
    width=1600,
    showlegend=True,
    plot_bgcolor="white",
    hovermode="closest",
    font=dict(size=11),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

# Update all x and y axes
fig.update_xaxes(
    title_text="AI Augmentation Potential",
    showgrid=False,
    showline=False,
    zeroline=False,
    title_font_size=12
)

fig.update_yaxes(
    title_text="Wagebill (Million $)",
    showgrid=False,
    showline=False,
    zeroline=False,
    title_font_size=12
)

fig.show()

print("\n" + "="*80)
print("✅ All scenario visualizations complete!")
print("="*80)
print("\n" + "="*80)


GENERATING SCENARIO-SPECIFIC VISUALIZATIONS (2x2 GRID)

Filtered to 41 DWAs with exposure >= 0.6 and wagebill >= $10M
✓ Top 5 DWAs - Total exposed wagebill: $259,170,069.02
✓ Top 10 DWAs - Total exposed wagebill: $407,942,557.64
✓ Top 15 DWAs - Total exposed wagebill: $529,116,334.45
✓ Top 20 DWAs - Total exposed wagebill: $623,259,252.90



✅ All scenario visualizations complete!



---
# SECTION 2: DWA Employee Drill-Down

Interactive tool to explore employees performing specific DWAs. Enter a DWA ID to see all employees and their statistics.

In [109]:
# === Load Employee-DWA Detailed Data ===
employees_path = BASE / "output/Tables/AAPL_employees_with_DWAs.parquet"
df_employees = pd.read_parquet(employees_path)

print(f"Loaded employee data: {len(df_employees):,} employee-DWA records")
print(f"Unique employees: {df_employees['user_id'].nunique():,}")
print(f"Unique DWAs: {df_employees['DWA ID'].nunique():,}")
print(f"\nAvailable columns: {', '.join(df_employees.columns.tolist())}")
print(f"Column data types:\n{df_employees.dtypes}")

Loaded employee data: 5,701,836 employee-DWA records
Unique employees: 267,174
Unique DWAs: 2,054

Available columns: user_id, position_id, region, company_raw, country, state, location_raw, metro_area, msa, city, remote_suitability, weight, title_raw, seniority, salary, description, onet_title, ultimate_parent_rcid, onet_code, prestige, highest_degree, sex_predicted, ethnicity_predicted, rn, O*NET-SOC Code, Title, DWA Title, DWA ID, dwa_time_normalized, exposure, dwa_weighted_salary
Column data types:
user_id                        Float64
position_id                    Float64
region                  string[python]
company_raw             string[python]
country                 string[python]
state                   string[python]
location_raw            string[python]
metro_area              string[python]
msa                     string[python]
city                    string[python]
remote_suitability             Float64
weight                         Float64
title_raw               

In [110]:
# === Interactive DWA Employee Explorer ===
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

def explore_dwa_employees(dwa_id):
    """
    Display all employees performing a specific DWA with descriptive statistics.
    
    Parameters:
    -----------
    dwa_id : str
        The DWA ID to explore (e.g., '4.A.2.a.3')
    """
    # Filter employees for this DWA
    dwa_employees = df_employees[df_employees['DWA ID'] == dwa_id].copy()
    
    if len(dwa_employees) == 0:
        print(f"❌ No employees found for DWA ID: {dwa_id}")
        print(f"\nAvailable DWA IDs: {sorted(df_employees['DWA ID'].unique())[:10]}...")
        return
    
    # Get DWA information
    dwa_info = aapl_dwa[aapl_dwa['DWA_ID'] == dwa_id].iloc[0] if dwa_id in aapl_dwa['DWA_ID'].values else None
    
    # Display header
    print("="*100)
    print(f"DWA EMPLOYEE ANALYSIS")
    print("="*100)
    print(f"\n📋 DWA ID: {dwa_id}")
    if dwa_info is not None:
        print(f"📝 Description: {dwa_info['DWA_TITLE']}")
        print(f"🎯 AI Exposure: {dwa_info['exposure']:.3f}")
        print(f"💰 Total Wagebill: ${dwa_info['DWA_WAGEBILL_TOTAL']:,.2f}")
    print(f"👥 Number of Employees: {len(dwa_employees):,}")
    
    # Descriptive Statistics
    print(f"\n{'='*100}")
    print("EMPLOYEE STATISTICS")
    print("="*100)
    
    # Salary statistics
    if 'salary' in dwa_employees.columns:
        print(f"\n💵 Salary Statistics:")
        print(f"   Total: ${dwa_employees['salary'].sum():,.2f}")
        print(f"   Mean: ${dwa_employees['salary'].mean():,.2f}")
        print(f"   Median: ${dwa_employees['salary'].median():,.2f}")
        print(f"   Std Dev: ${dwa_employees['salary'].std():,.2f}")
        print(f"   Min: ${dwa_employees['salary'].min():,.2f}")
        print(f"   Max: ${dwa_employees['salary'].max():,.2f}")
    
    # Time allocation statistics
    if 'dwa_time_normalized' in dwa_employees.columns:
        print(f"\n⏱️  Time Allocation Statistics (% of work time on this DWA):")
        print(f"   Mean: {dwa_employees['dwa_time_normalized'].mean()*100:.1f}%")
        print(f"   Median: {dwa_employees['dwa_time_normalized'].median()*100:.1f}%")
        print(f"   Min: {dwa_employees['dwa_time_normalized'].min()*100:.1f}%")
        print(f"   Max: {dwa_employees['dwa_time_normalized'].max()*100:.1f}%")
    
    # Occupation distribution
    if 'onet_code' in dwa_employees.columns and 'job_title' in dwa_employees.columns:
        print(f"\n👔 Top 10 Occupations performing this DWA:")
        occ_dist = dwa_employees.groupby(['onet_code', 'job_title']).size().sort_values(ascending=False).head(10)
        for idx, ((onet, title), count) in enumerate(occ_dist.items(), 1):
            print(f"   {idx:2d}. {title} ({onet}): {count:,} employees")
    
    # Display employee list (top 50)
    print(f"\n{'='*100}")
    print(f"EMPLOYEE LIST (Top 50 by Salary)")
    print("="*100)
    
    # Select relevant columns and sort
    display_cols = ['user_id', 'onet_code', 'onet_title', 'salary']
    
    # Add demographic and work information if available
    optional_cols = ['sex_predicted', 'ethnicity_predicted', 'city', 'seniority', 'dwa_time_normalized', 'dwa_weighted_salary']
    for col in optional_cols:
        if col in dwa_employees.columns:
            display_cols.append(col)
    
    # Filter to available columns
    display_cols = [col for col in display_cols if col in dwa_employees.columns]
    
    employee_list = dwa_employees[display_cols].sort_values('salary', ascending=False).head(50)
    
    # Format for display
    if 'dwa_time_normalized' in employee_list.columns:
        employee_list['dwa_time_pct'] = (employee_list['dwa_time_normalized'] * 100).round(1)
        employee_list = employee_list.drop(columns=['dwa_time_normalized'])
    
    # Rename columns for clarity
    rename_dict = {
        'user_id': 'Employee_ID',
        'onet_code': 'ONET_Code',
        'job_title': 'Job_Title',
        'salary': 'Salary',
        'gender': 'Gender',
        'ethnicity': 'Ethnicity',
        'location': 'Location',
        'seniority': 'Seniority',
        'dwa_weighted_salary': 'DWA_Allocated_Salary',
        'dwa_time_pct': 'Time_on_DWA_%'
    }
    employee_list = employee_list.rename(columns={k: v for k, v in rename_dict.items() if k in employee_list.columns})
    
    # Display styled dataframe
    display(employee_list.style.format({
        'Salary': '${:,.2f}',
        'DWA_Allocated_Salary': '${:,.2f}' if 'DWA_Allocated_Salary' in employee_list.columns else None,
        'Time_on_DWA_%': '{:.1f}%' if 'Time_on_DWA_%' in employee_list.columns else None
    }).set_properties(**{
        'text-align': 'left',
        'font-size': '11pt'
    }).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-weight', 'bold'), ('background-color', '#f0f0f0')]}
    ]))
    
    print(f"\n✅ Showing {len(employee_list):,} of {len(dwa_employees):,} total employees")
    print("="*100)

# Example usage - you can change the DWA ID below
print("\n" + "="*100)
print("HOW TO USE:")
print("="*100)
print("Call the function: explore_dwa_employees('DWA_ID')")
print("Example: explore_dwa_employees('4.A.2.a.3')")
print("\nAvailable DWA IDs from the bubble chart above.")
print("="*100)


HOW TO USE:
Call the function: explore_dwa_employees('DWA_ID')
Example: explore_dwa_employees('4.A.2.a.3')

Available DWA IDs from the bubble chart above.


In [111]:
# === EXAMPLE: Explore a specific DWA ===
# First, let's see what DWA IDs are available

# print("\nTop 20 DWAs by wagebill (copy a DWA ID to explore):")
# top_dwas = aapl_dwa.nlargest(20, 'DWA_WAGEBILL_TOTAL')[['DWA_ID', 'DWA_TITLE', 'exposure', 'position_count']]
# for idx, row in top_dwas.iterrows():
#     print(f"  {row['DWA_ID']:<15} - {row['DWA_TITLE'][:70]:<70} | Exposure: {row['exposure']:.3f} | Employees: {row['position_count']:,}")

# print("\n" + "="*100)
# print("To explore a DWA, run: explore_dwa_employees('DWA_ID')")
# print("Example: explore_dwa_employees('4.A.4.a.2')")
# print("="*100)

In [112]:
# === EXPLORE A SPECIFIC DWA ===
# Set the DWA ID once - it will be used by both functions below
DWA_ID = '4.A.1.a.1.I01.D01'

explore_dwa_employees(DWA_ID)

DWA EMPLOYEE ANALYSIS

📋 DWA ID: 4.A.1.a.1.I01.D01
📝 Description: Review art or design materials.
🎯 AI Exposure: 0.762
💰 Total Wagebill: $87,539,427.13
👥 Number of Employees: 6,115

EMPLOYEE STATISTICS

💵 Salary Statistics:
   Total: $677,136,379.68
   Mean: $110,788.02
   Median: $90,192.24
   Std Dev: $68,117.82
   Min: $12,744.43
   Max: $462,775.71

⏱️  Time Allocation Statistics (% of work time on this DWA):
   Mean: 12.5%
   Median: 14.4%
   Min: 1.4%
   Max: 14.4%

EMPLOYEE LIST (Top 50 by Salary)


,Employee_ID,ONET_Code,onet_title,Salary,sex_predicted,ethnicity_predicted,city,Seniority,DWA_Allocated_Salary,Time_on_DWA_%
3541220,588957512.000000,27-1011.00,Art Directors,"$462,775.71",M,White,Cupertino,6,"$66,661.53",14.4%
2983850,489567069.000000,27-1011.00,Art Directors,"$455,474.35",M,White,San Francisco,6,"$65,609.79",14.4%
3395021,562523471.000000,27-1011.00,Art Directors,"$441,721.90",F,Black,San Francisco,5,"$63,628.79",14.4%
159742,25623044.000000,27-1011.00,Art Directors,"$434,700.83",M,White,Sunnyvale,6,"$62,617.42",14.4%
5464859,2207349530.000000,27-1011.00,Art Directors,"$413,294.01",M,White,,6,"$59,533.83",14.4%
1013981,159397224.000000,27-1011.00,Art Directors,"$413,217.50",M,White,Cupertino,4,"$59,522.81",14.4%
621614,97683376.000000,27-1011.00,Art Directors,"$406,590.62",M,White,Cupertino,5,"$58,568.23",14.4%
631538,99216378.000000,27-1011.00,Art Directors,"$394,754.49",F,White,Cupertino,5,"$56,863.27",14.4%
2391593,388013607.000000,27-1011.00,Art Directors,"$394,660.36",M,White,San Francisco,5,"$56,849.71",14.4%
394719,61991578.000000,27-1011.00,Art Directors,"$389,909.06",M,API,Cupertino,5,"$56,165.30",14.4%



✅ Showing 50 of 6,115 total employees


In [113]:
# === Visualize DWA Demographics and Statistics ===
def visualize_dwa_demographics(dwa_id):
    
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    
    # Filter employees for this DWA
    dwa_employees = df_employees[df_employees['DWA ID'] == dwa_id].copy()
    
    if len(dwa_employees) == 0:
        print(f"No employees found for DWA ID: {dwa_id}")
        return
    
    # Get DWA information
    dwa_info = (
        aapl_dwa[aapl_dwa['DWA_ID'] == dwa_id].iloc[0]
        if dwa_id in aapl_dwa['DWA_ID'].values
        else None
    )
    dwa_title = dwa_info['DWA_TITLE'][:60] if dwa_info is not None else str(dwa_id)
    
    # Create 2x2 subplots
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[
            "Gender Distribution",
            "Ethnicity Breakdown",
            "Salary Statistics",
            "Top 5 Locations",
        ],
        specs=[
            [{"type": "pie"}, {"type": "bar"}],
            [{"type": "bar"}, {"type": "bar"}],
        ],
        horizontal_spacing=0.15,
        vertical_spacing=0.15,
    )
    
    # === 1. Gender Distribution ===
    gender_col = "sex_predicted" if "sex_predicted" in dwa_employees.columns else "gender"
    if gender_col in dwa_employees.columns:
        gender_counts = dwa_employees[gender_col].value_counts()
        fig.add_trace(
            go.Pie(
                labels=gender_counts.index,
                values=gender_counts.values,
                marker=dict(colors=["#004c99", "#9bc2e6"]),
                textinfo="label+percent",
                showlegend=False,
            ),
            row=1,
            col=1,
        )
    
    # === 2. Ethnicity Breakdown ===
    ethnicity_col = (
        "ethnicity_predicted"
        if "ethnicity_predicted" in dwa_employees.columns
        else "ethnicity"
    )
    if ethnicity_col in dwa_employees.columns:
        ethnicity_counts = dwa_employees[ethnicity_col].value_counts().head(10)
        fig.add_trace(
            go.Bar(
                x=ethnicity_counts.values,
                y=ethnicity_counts.index,
                orientation="h",
                marker_color="#3d8bd9",
                text=ethnicity_counts.values,
                textposition="auto",
                showlegend=False,
            ),
            row=1,
            col=2,
        )
    
    # === 3. Salary Statistics (BLUE ONLY) ===
    if "salary" in dwa_employees.columns:
        categories = ["Average", "Maximum", "Minimum"]
        salary_vals = [
            dwa_employees["salary"].mean(),
            dwa_employees["salary"].max(),
            dwa_employees["salary"].min(),
        ]
        
        fig.add_trace(
            go.Bar(
                x=categories,
                y=salary_vals,
                marker_color="#004c99",
                text=[
                    f"${v/1e6:.1f}M" if v >= 1e6 else f"${v/1e3:.0f}K"
                    for v in salary_vals
                ],
                textposition="auto",
                showlegend=False,
            ),
            row=2,
            col=1,
        )
    
    # === 4. Top 5 Locations ===
    location_col = "city" if "city" in dwa_employees.columns else "location"
    if location_col in dwa_employees.columns:
        location_counts = dwa_employees[location_col].value_counts().head(5)
        colors = ["#004c99", "#0066cc", "#3385d6", "#5a9fd4", "#9bc2e6"]
        
        fig.add_trace(
            go.Bar(
                x=location_counts.values,
                y=location_counts.index,
                orientation="h",
                marker=dict(color=colors[: len(location_counts)]),
                text=location_counts.values,
                textposition="auto",
                showlegend=False,
            ),
            row=2,
            col=2,
        )
    
    # === LAYOUT: REMOVE GRAY PANELS ===
    fig.update_layout(
        template="plotly_white",        # ← critical
        paper_bgcolor="white",
        plot_bgcolor="white",
        title_text=(
            f"DWA Demographics & Statistics: {dwa_title}"
            f"<br><sub>DWA ID: {dwa_id} | Total Employees: {len(dwa_employees):,}</sub>"
        ),
        title_x=0.5,
        height=900,
        width=1600,
    )
    
    # === AXES ===
    fig.update_xaxes(showgrid=True, gridcolor="#eaeaea")
    fig.update_yaxes(showgrid=True, gridcolor="#eaeaea")
    
    fig.update_xaxes(title_text="Number of Employees", row=1, col=2)
    fig.update_xaxes(title_text="Salary Category", row=2, col=1)
    fig.update_yaxes(title_text="Salary ($)", row=2, col=1)
    fig.update_xaxes(title_text="Number of Employees", row=2, col=2)
    
    fig.show()

In [114]:
# === CREATE DASHBOARD FOR SPECIFIC DWA ===
# Visualize the same DWA in a McKinsey-style dashboard

visualize_dwa_demographics(DWA_ID)